<a href="https://colab.research.google.com/github/2026SD4/26SD4_19_HATANAKA_MAO/blob/main/SD4AI%E6%BC%94%E7%BF%9204%E7%95%91%E4%B8%AD%E6%94%BF%E5%A4%AE_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U -q google-generativeai google-ai-generativelanguage
import google.generativeai as genai
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
import json
def get_current_weather(location, unit="fahrenheit"):
  """指定された場所の現在の天気を取得する"""
  if "tokyo" in location.lower():
    return json.dumps({
        "location":"tokyo",
        "temperature":"10",
        "unit": "celsius"
    }, ensure_ascii=False)
  elif "san francisco" in location.lower():
    return json.dumps({
        "location":"San Francisco",
        "temperature":"72",
        "unit": "fahrenheit"
    }, ensure_ascii=False)

In [ ]:
import google.generativeai as genai

model = genai.GenerativeModel(
    model_name='gemini-3.6-flash',
    tools=[get_current_weather],
    tool_config={'function_calling_config':{'mode':'AUTO'}}
)

chat = model.start_chat()

response = chat.send_message("東京の気温は30度よりも上ですか？",
                             tool_config={'function_calling_config':{'mode':'ANY'}})

print("function callの結果：実行する関数名と引数が指定されている")
print(response.candidates[0].content.parts[0].function_call)

function callの結果：実行する関数名と引数が指定されている
name: "get_current_weather"
args {
  fields {
    key: "location"
    value {
      string_value: "Tokyo"
    }
  }
}
id: "call_821269"



In [ ]:
function_call = response.candidates[0].content.parts[0].function_call
if function_call:
  #関数を実行
  function_args = function_call.args
  function_response_str = get_current_weather(
      location=function_args['location'],
      unit=function_args.get('unit')
  )
  print(f"ローカル関数呼び出しの結果：\n{function_response_str}")

  # 辞書を使ったすっきりとした書き方
  second_response = chat.send_message({
      "function_response":{
          "id":function_call.id,
          "name":function_call.name,
          "response":{"result":function_response_str}
      }
  })
  print(f"２度目の呼び出しの結果応答:{second_response}")
  print(f"応答に含まれたメッセージ:{second_response.text}")